|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Reading through the table<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: write the paged attention oracle<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import math
import torch
import torch.nn.functional as F

import cudalib
from tests.helpers import build_paged, random_kv

Write the reference implementation of paged attention.

This is stage 07. Speed is not the point. This function is the **oracle**. You
check every later kernel against it.

It must stay correct on a shuffled pool. It must stay correct on [ragged](../../GLOSSARY.md#flat-batch)
lengths. It must stay correct when the unused slots hold another request's
tokens.

Make it correct, and accept that it is slow. Stage 08 recovers the speed.

In [ ]:
### run this cell

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

num_seqs, num_heads, num_kv_heads, head_dim, context_len = 4, 8, 2, 64, 100
BLOCK_SIZE = 16

keys, values = random_kv(num_seqs, num_kv_heads, context_len, head_dim, device)
query = torch.randn(num_seqs, num_heads, head_dim, device=device)
key_cache, value_cache, block_tables, context_lens = build_paged(keys, values, BLOCK_SIZE)

print(f'pool {tuple(key_cache.shape)}, block table {tuple(block_tables.shape)}')
print(f'sequence 0 lives in blocks {block_tables[0].tolist()}')

# Exercise 1: the dense oracle first

Use no paging. Compute attention over the original contiguous tensors. This
gives you something to check the paged version against.

In [ ]:
def reference_attention(query, keys, values, scale=None):
  """No paging. q (S,H,D), K/V (S,KVH,L,D). This is the oracle."""
  num_seqs, num_heads, head_dim = query.shape
  group = num_heads // keys.shape[1]        # query heads per KV head
  scale = scale or 1.0/math.sqrt(head_dim)
  output = torch.empty_like(query)
  for seq in range(num_seqs):
    for head in range(num_heads):
      # which KV head does query head h read?
      scores = 
      output[seq,head] = 
  return output

expected = reference_attention(query, keys, values)
print('oracle:', tuple(expected.shape))

# Exercise 2: the scatter

Attention cannot read the pool until something writes it. You receive one flat
slot per token. Put K and V in the correct places.

In [ ]:
def write_kv(key_cache, value_cache, key, value, slot_indices):
  """Scatter this step's K/V into the pool.
  key/value (T, KVH, D), slot_indices (T,) flat slots from slot_index().

  A flat slot decomposes as block = slot // block_size, off = slot % block_size.
  This is the 'slot mapping' you will see all over serving code."""
  block_size = key_cache.shape[2]
  for token, slot in enumerate(slot_indices.tolist()):
    

written_keys = torch.zeros_like(key_cache)
written_values = torch.zeros_like(value_cache)
slots = torch.tensor([block_tables[0,0]*BLOCK_SIZE + 0, block_tables[0,0]*BLOCK_SIZE + 1])
write_kv(written_keys, written_values, keys[0,:, :2].permute(1,0,2), values[0,:, :2].permute(1,0,2), slots)
print('wrote 2 tokens; matches source:',
      torch.allclose(written_keys[block_tables[0,0], :, :2], keys[0,:, :2]))

# Exercise 3: the gather

Walk the block table. Collect the blocks. Cut the result to `context_len`.
Then do the arithmetic from Exercise 1.

In [ ]:
def paged_attention(query, key_cache, value_cache, block_tables, context_lens, scale=None):
  num_seqs, num_heads, head_dim = query.shape
  num_kv_heads, block_size = key_cache.shape[1], key_cache.shape[2]
  group = num_heads // num_kv_heads
  scale = scale or 1.0/math.sqrt(head_dim)
  output = torch.empty_like(query)

  for seq in range(num_seqs):
    context_len = int(context_lens[seq])

    # which physical blocks hold this sequence's first n tokens?
    blocks = 

    # gather them into (KVH, n, D). Careful with the axis order:
    # kc[blocks] is (nblocks, KVH, BS, D) and you want KVH first.
    seq_keys = 
    seq_values = 

    for head in range(num_heads):
      kv_head = head // group
      scores = 
      output[seq,head] = 
  return output

output = paged_attention(query, key_cache, value_cache, block_tables, context_lens)
print('max difference from the oracle:', (output - expected).abs().max().item())

# Exercise 4: two quiet failures

The allocator recycles blocks, so the slots past `context_len` hold another
request's tokens. And no two sequences have the same length.

In [ ]:
# poison every slot past context_len, the way a recycled block would be
poisoned_keys, poisoned_values = key_cache.clone(), value_cache.clone()
for seq in range(num_seqs):
  for block in range(block_tables.shape[1]):
    block_id = int(block_tables[seq,block])
    for offset in range(BLOCK_SIZE):
      if block*BLOCK_SIZE + offset >= context_len:
        

poisoned_output = paged_attention(query, poisoned_keys, poisoned_values, block_tables, context_lens)
print('output unchanged:', torch.allclose(output, poisoned_output, atol=1e-5))

# and every sequence may be a different length
ragged = torch.tensor([100, 1, 17, 64], dtype=torch.int32, device=device)
ragged_output = paged_attention(query, key_cache, value_cache, block_tables, ragged)
print('ragged context lengths ran:', tuple(ragged_output.shape))

# Exercise 5: what did paging cost?

In [ ]:
if device == 'cuda':
  num_seqs, num_heads, num_kv_heads, head_dim, context_len = 32, 16, 8, 128, 512
  bench_keys, bench_values = random_kv(num_seqs, num_kv_heads, context_len, head_dim, device, dtype=torch.float16)
  bench_query = torch.randn(num_seqs, num_heads, head_dim, device=device, dtype=torch.float16)
  bench_key_cache, bench_value_cache, bench_block_tables, bench_context_lens = build_paged(bench_keys, bench_values, BLOCK_SIZE)

  paged = 
  dense = 
  print(f'contiguous SDPA: {dense:8.3f} ms')
  print(f'your paged loop: {paged:8.3f} ms   ({paged/dense:.0f}x slower)')

### Before you open the solution

1. Your paged version runs much slower than SDPA. Name the two separate
   reasons. You wrote both of them.
2. `key_cache[blocks]` makes a new tensor. How large is it, for 32 sequences of 512
   tokens? Compare that size with the size of the output you want.
3. Exercise 4 poisoned the slots past `context_len`, and your output did not
   move. Which line of your gather protects you? What happens if you delete
   it?